In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [3]:
spark = SparkSession.builder.appName("NYC").getOrCreate()
base_path = "/home/jovyan/work"

In [7]:
# df = spark.read.parquet(f"{base_path}/weather_partitioned/year=2020/month=1/f646403d9ff74adeb8c831600daa6810-0.parquet")
df = spark.read.parquet(f"{base_path}/weather_partitioned/")

df_filtered = df.filter((df.year == 2020) & (df.month == 1))

In [5]:
#TODO: change filter to not only one month but entire year or more

In [8]:
df.printSchema()

root
 |-- STATION: string (nullable = true)
 |-- NAME: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- ELEVATION: double (nullable = true)
 |-- DATE: timestamp_ntz (nullable = true)
 |-- REPORT_TYPE: string (nullable = true)
 |-- SOURCE: double (nullable = true)
 |-- DailyAverageDryBulbTemperature: double (nullable = true)
 |-- DailyMaximumDryBulbTemperature: double (nullable = true)
 |-- DailyMinimumDryBulbTemperature: double (nullable = true)
 |-- DailyPrecipitation: string (nullable = true)
 |-- DailyWeather: string (nullable = true)
 |-- HourlyDryBulbTemperature: double (nullable = true)
 |-- HourlyPrecipitation: string (nullable = true)
 |-- Sunrise: double (nullable = true)
 |-- Sunset: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [9]:
df_filtered.show(3, vertical=True)

-RECORD 0----------------------------------------------
 STATION                        | USW00094728          
 NAME                           | NY CITY CENTRAL P... 
 LATITUDE                       | 40.77898             
 LONGITUDE                      | -73.96925            
 ELEVATION                      | 42.7                 
 DATE                           | 2020-01-01 00:00:00  
 REPORT_TYPE                    | SOD                  
 SOURCE                         | NULL                 
 DailyAverageDryBulbTemperature | 3.1                  
 DailyMaximumDryBulbTemperature | 5.0                  
 DailyMinimumDryBulbTemperature | 1.1                  
 DailyPrecipitation             | 0.0                  
 DailyWeather                   |                      
 HourlyDryBulbTemperature       | NULL                 
 HourlyPrecipitation            |                      
 Sunrise                        | 720.0                
 Sunset                         | 1638.0        

In [10]:
# --- remove unnecessary cols ---
# --- copy vals to empty rows for each day ---

df_filtered = df_filtered.drop("STATION", "NAME", "LATITUDE", "LONGITUDE", "ELEVATION", "SOURCE")

df_filtered.show(3, vertical=True)

-RECORD 0---------------------------------------------
 DATE                           | 2020-01-01 00:00:00 
 REPORT_TYPE                    | SOD                 
 DailyAverageDryBulbTemperature | 3.1                 
 DailyMaximumDryBulbTemperature | 5.0                 
 DailyMinimumDryBulbTemperature | 1.1                 
 DailyPrecipitation             | 0.0                 
 DailyWeather                   |                     
 HourlyDryBulbTemperature       | NULL                
 HourlyPrecipitation            |                     
 Sunrise                        | 720.0               
 Sunset                         | 1638.0              
 year                           | 2020                
 month                          | 1                   
-RECORD 1---------------------------------------------
 DATE                           | 2020-01-01 00:51:00 
 REPORT_TYPE                    | FM-15               
 DailyAverageDryBulbTemperature | NULL                
 DailyMaxi

In [17]:
df_weather = df_filtered.withColumn("date_only", to_date(col("DATE")))
df_hourly = df_weather.filter(col("REPORT_TYPE") != "SOD")

df_daily_summary = df_weather.filter(col("REPORT_TYPE") == "SOD").select(
    col("date_only").alias("summary_date"),
    col("DailyAverageDryBulbTemperature").alias("daily_avg_temp"),
    col("DailyMaximumDryBulbTemperature").alias("daily_max_temp"),
    col("DailyMinimumDryBulbTemperature").alias("daily_min_temp"),
    col("DailyPrecipitation").alias("daily_precipitation"),
    col("Sunrise").alias("daily_sunrise"),
    col("Sunset").alias("daily_sunset")
)

df_final_weather = df_hourly.join(
    broadcast(df_daily_summary),
    df_hourly.date_only == df_daily_summary.summary_date,
    "left"
).drop("summary_date")

#df_final_weather.show(5, vertical=True)
df_final_weather.select(
    "DATE", 
    "REPORT_TYPE", 
    "daily_avg_temp", 
    "daily_precipitation"
).show(5)

+-------------------+-----------+--------------+-------------------+
|               DATE|REPORT_TYPE|daily_avg_temp|daily_precipitation|
+-------------------+-----------+--------------+-------------------+
|2020-01-01 00:51:00|      FM-15|           3.1|                0.0|
|2020-01-01 01:51:00|      FM-15|           3.1|                0.0|
|2020-01-01 02:51:00|      FM-15|           3.1|                0.0|
|2020-01-01 03:51:00|      FM-15|           3.1|                0.0|
|2020-01-01 04:51:00|      FM-15|           3.1|                0.0|
+-------------------+-----------+--------------+-------------------+
only showing top 5 rows



In [ ]:
#TODO: drop DailyAverageDryBulbTemperature, ... as they contain not a lot of values or leave them be anyway?

In [18]:
df_final_weather.sort("DATE", ascending=True).show(3, vertical=True)

-RECORD 0---------------------------------------------
 DATE                           | 2020-01-01 00:51:00 
 REPORT_TYPE                    | FM-15               
 DailyAverageDryBulbTemperature | NULL                
 DailyMaximumDryBulbTemperature | NULL                
 DailyMinimumDryBulbTemperature | NULL                
 DailyPrecipitation             |                     
 DailyWeather                   |                     
 HourlyDryBulbTemperature       | 4.4                 
 HourlyPrecipitation            | 0.0                 
 Sunrise                        | NULL                
 Sunset                         | NULL                
 year                           | 2020                
 month                          | 1                   
 date_only                      | 2020-01-01          
 daily_avg_temp                 | 3.1                 
 daily_max_temp                 | 5.0                 
 daily_min_temp                 | 1.1                 
 daily_pre

In [98]:
spark.stop()